# Train YOLOv8/v11 -- Nhan dien phuong tien giao thong Viet Nam

**De tai:** Ung dung hoc sau trong nhan dien phuong tien va uoc tinh mat do giao thong  
**Sinh vien:** Nguyen Huynh -- 102220024  
**GVHD:** TS. Trinh Cong Duy

---

### Huong dan su dung
1. **Runtime -> Change runtime type -> T4 GPU**
2. Chay tung cell theo thu tu (Shift+Enter)
3. Upload anh vao Google Drive truoc khi chay **Phan 4**

| Class ID | Ten | Ghi chu |
|----------|-----|---------|
| 0 | xe_may | ~70% phuong tien do thi VN |
| 1 | o_to | |
| 2 | xe_tai | |
| 3 | xe_bus | |

## Phan 1 -- Kiem tra GPU & Moi truong

In [1]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    for line in gpu_info.stdout.split('\n')[:12]:
        print(line)
    print('\n GPU kha dung!')
else:
    print('Khong tim thay GPU!')
    print('Vao Runtime -> Change runtime type -> chon T4 GPU')

import psutil
ram = psutil.virtual_memory()
print(f'RAM tong: {ram.total/1e9:.1f} GB | RAM trong: {ram.available/1e9:.1f} GB')
print(f'Python: {sys.version.split()[0]}')

Tue Apr 14 22:26:55 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 516.40       Driver Version: 516.40       CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   52C    P0    13W /  N/A |      0MiB /  4096MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+

 GPU kha dung!
RAM tong: 16.8 GB | RAM trong: 3.5 GB
Python: 3.14.3


## Phan 2 -- Cai dat thu vien

In [ ]:
%%time
print('Dang cai dat (khoang 1-2 phut)...')

import subprocess, sys
for pkg in ['ultralytics>=8.3.0', 'albumentations>=1.4.0', 'opencv-python']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import torch
from ultralytics import YOLO
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(' Cai dat hoan tat!')

## Phan 3 -- Import & Cau hinh

In [ ]:
import os, yaml, shutil, random, logging, time
from pathlib import Path
from datetime import datetime

import cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.notebook import tqdm
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED); import numpy as np; np.random.seed(SEED)
torch.manual_seed(SEED)
logging.basicConfig(level=logging.WARNING)

print(' Import thanh cong!')

In [ ]:
# ================================================================
#   CHINH SUA CAU HINH O DAY
# ================================================================
CONFIG = {
    # --- Model ---
    # Chon 1 option:
    # 'yolov8n.pt'  nano   (nhanh nhat, mAP thap hon)
    # 'yolov8s.pt'  small
    # 'yolov8m.pt'  medium (khuyen nghi cho de tai)
    # 'yolov8l.pt'  large  (can nhieu VRAM hon)
    # 'yolo11m.pt'  YOLOv11 medium
    'model_name': 'yolov8m.pt',

    # --- Dataset ---
    'num_classes': 4,
    'class_names': ['xe_may', 'o_to', 'xe_tai', 'xe_bus'],

    # --- Training ---
    'epochs':    100,  # giam xuong 50 neu muon test nhanh
    'patience':   20,  # early stopping
    'batch_size': 16,  # giam xuong 8 neu loi out of memory
    'img_size':  640,
    'lr0':      0.01,
    'lrf':     0.001,

    # --- Augmentation ---
    'mosaic':  1.0,
    'mixup':   0.1,
    'fliplr':  0.5,
    'degrees': 5.0,
    'scale':   0.5,
    'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,

    # --- Google Drive paths ---
    'drive_data_path': '/content/drive/MyDrive/DATN_Data',

    # --- Output ---
    'project':  '/content/runs/detect',
    'exp_name': f'vn_traffic_{datetime.now().strftime("%Y%m%d_%H%M")}',
}

CLASS_COLORS = {
    0: (255,165,  0),  # cam:  xe_may
    1: (  0,120,255),  # xanh: o_to
    2: (255, 50, 50),  # do:   xe_tai
    3: ( 50,200, 50),  # la:   xe_bus
}

print(' Cau hinh OK:')
for k, v in CONFIG.items():
    print(f'   {k}: {v}')

## Phan 4 -- Ket noi Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

base = Path(CONFIG['drive_data_path'])
if base.exists():
    n_imgs = len(list((base / 'images' / 'train').glob('*.jpg')) +
                 list((base / 'images' / 'train').glob('*.png')))
    print(f' Data co san: {n_imgs} anh train')
else:
    # Tao cau truc thu muc
    for split in ['train', 'val', 'test']:
        (base / 'images' / split).mkdir(parents=True, exist_ok=True)
        (base / 'labels' / split).mkdir(parents=True, exist_ok=True)
    print(f' Da tao thu muc tai: {base}')
    print()
    print('Cau truc thu muc tren Drive:')
    print(f'  {base}/')
    print(f'    images/')
    print(f'      train/  <-- upload anh .jpg/.png vao day (70%)')
    print(f'      val/    <-- 15%')
    print(f'      test/   <-- 15%')
    print(f'    labels/')
    print(f'      train/  <-- file .txt annotation (cung ten voi anh)')
    print(f'      val/')
    print(f'      test/')
    print()
    print('Format annotation (moi dong trong .txt):')
    print('  <class_id> <x_center> <y_center> <width> <height>')
    print('  (tat ca normalize ve [0.0, 1.0])')

## Phan 5 -- Chia dataset (neu chua chia)

In [ ]:
# Dung cell nay neu anh chua duoc chia train/val/test
# Dieu kien: tat ca anh va annotation de trong 2 thu muc rieng

RAW_IMAGES = f"{CONFIG['drive_data_path']}/raw/images"  # sua o day
RAW_LABELS = f"{CONFIG['drive_data_path']}/raw/labels"  # sua o day

def split_dataset(images_dir, labels_dir, output_dir,
                  train=0.70, val=0.15, test=0.15, seed=42):
    imgs_dir = Path(images_dir); lbls_dir = Path(labels_dir)
    out_dir  = Path(output_dir)
    EXTS = {'.jpg','.jpeg','.png','.bmp'}

    all_imgs = sorted(f for f in imgs_dir.iterdir() if f.suffix.lower() in EXTS)
    pairs = [(img, lbls_dir / (img.stem + '.txt'))
             for img in all_imgs
             if (lbls_dir / (img.stem + '.txt')).exists()]

    random.seed(seed); random.shuffle(pairs)
    n = len(pairs); n_tr = int(n*train); n_v = int(n*val)
    splits = {
        'train': pairs[:n_tr],
        'val':   pairs[n_tr:n_tr+n_v],
        'test':  pairs[n_tr+n_v:],
    }
    for name, ps in splits.items():
        (out_dir/'images'/name).mkdir(parents=True, exist_ok=True)
        (out_dir/'labels'/name).mkdir(parents=True, exist_ok=True)
        for img, lbl in tqdm(ps, desc=f'  {name}'):
            shutil.copy2(img, out_dir/'images'/name/img.name)
            shutil.copy2(lbl, out_dir/'labels'/name/lbl.name)
        print(f'  {name}: {len(ps)} anh')
    print(f'Hoan tat: {n} anh -> {out_dir}')

# Bo comment dong duoi de chay:
# split_dataset(RAW_IMAGES, RAW_LABELS, CONFIG['drive_data_path'])
print('Cell san sang. Bo comment dong cuoi va chay neu can chia dataset.')

## Phan 6 -- Tao dataset.yaml

In [ ]:
DATA_YAML = '/content/dataset.yaml'

yaml_cfg = {
    'path':  CONFIG['drive_data_path'],
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    CONFIG['num_classes'],
    'names': CONFIG['class_names'],
}
import yaml
with open(DATA_YAML, 'w') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False)

print(' dataset.yaml:')
with open(DATA_YAML) as f: print(f.read())

## Phan 7 -- Kiem tra dataset

In [ ]:
def verify_dataset(data_yaml):
    import yaml
    with open(data_yaml) as f: cfg = yaml.safe_load(f)
    data_path = Path(cfg['path']); nc = cfg['nc']
    print(f'{"Split":<6} {"Anh":>6} {"Bbox":>7}  ' +
          '  '.join(f'{n[:8]:>8}' for n in cfg['names']))
    print('-'*70)
    for split in ['train','val','test']:
        img_dir = data_path / cfg.get(split, f'images/{split}')
        lbl_dir = Path(str(img_dir).replace('images','labels'))
        if not img_dir.exists():
            print(f'  {split}: khong tim thay {img_dir}'); continue
        imgs = sorted(f for f in img_dir.iterdir()
                      if f.suffix.lower() in {'.jpg','.jpeg','.png'})
        cc = [0]*nc; missing=0
        for img in imgs:
            lbl = lbl_dir/(img.stem+'.txt')
            if not lbl.exists(): missing+=1; continue
            for line in lbl.read_text().strip().splitlines():
                p = line.strip().split()
                if len(p)==5:
                    cid=int(p[0])
                    if 0<=cid<nc: cc[cid]+=1
        tb = sum(cc)
        cs = '  '.join(f'{c:>8}' for c in cc)
        warn = ' (missing labels!)' if missing else ''
        print(f'{split:<6} {len(imgs):>6} {tb:>7}  {cs}{warn}')
    print()
    print(' Phan phoi class OK!' if True else '')

verify_dataset(DATA_YAML)

## Phan 8 -- Xem thu anh + annotation

In [ ]:
def visualize_samples(data_yaml, split='train', n=9):
    import yaml
    with open(data_yaml) as f: cfg = yaml.safe_load(f)
    data_path = Path(cfg['path'])
    img_dir = data_path / cfg.get(split, f'images/{split}')
    lbl_dir = Path(str(img_dir).replace('images','labels'))
    imgs = sorted(f for f in img_dir.iterdir()
                  if f.suffix.lower() in {'.jpg','.jpeg','.png'})
    if not imgs: print('Khong co anh.'); return
    sampled = random.sample(imgs, min(n, len(imgs)))
    cols=3; rows=(len(sampled)+2)//3
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows*4))
    axes = np.array(axes).flatten()
    for ax, img_path in zip(axes, sampled):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        ax.imshow(img)
        lbl = lbl_dir/(img_path.stem+'.txt')
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                p=line.strip().split()
                if len(p)!=5: continue
                cid=int(p[0])
                xc,yc,bw,bh=map(float,p[1:])
                x1=(xc-bw/2)*w; y1=(yc-bh/2)*h
                col=[c/255 for c in CLASS_COLORS.get(cid,(200,200,200))]
                ax.add_patch(patches.Rectangle((x1,y1),bw*w,bh*h,
                    lw=2,edgecolor=col,facecolor='none'))
                ax.text(x1,y1-4,cfg['names'][cid],fontsize=7,color=col,
                    bbox=dict(boxstyle='round,pad=0.1',facecolor='white',alpha=0.6))
        ax.set_title(img_path.name[:25],fontsize=8); ax.axis('off')
    for ax in axes[len(sampled):]: ax.axis('off')
    plt.suptitle(f'Preview: {split.upper()}', fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

visualize_samples(DATA_YAML, split='train', n=9)

## Phan 9 -- TRAINING

> **Thoi gian uoc tinh tren Colab T4:**
> - `yolov8n` x 100 epochs ~ 30-45 phut
> - `yolov8m` x 100 epochs ~ 1.5-2.5 gio
> - `yolov8l` x 100 epochs ~ 3-5 gio
>
> Colab co the ngat ket noi sau ~90 phut khong hoat dong.
> Weights duoc luu tu dong vao Drive moi 10 epochs (neu da cai).

In [ ]:
model = YOLO(CONFIG['model_name'])
model.info(verbose=False)
device = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {"GPU -- " + torch.cuda.get_device_name(0) if device=="0" else "CPU"}')
print(f'Model:  {CONFIG["model_name"]}')

In [ ]:
print('Bat dau training...')
print(f'Model: {CONFIG["model_name"]} | Epochs: {CONFIG["epochs"]} | Batch: {CONFIG["batch_size"]}')

results = model.train(
    data         = DATA_YAML,
    epochs       = CONFIG['epochs'],
    patience     = CONFIG['patience'],
    batch        = CONFIG['batch_size'],
    imgsz        = CONFIG['img_size'],
    device       = device,
    workers      = 2,
    seed         = SEED,
    lr0          = CONFIG['lr0'],
    lrf          = CONFIG['lrf'],
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,
    mosaic       = CONFIG['mosaic'],
    mixup        = CONFIG['mixup'],
    fliplr       = CONFIG['fliplr'],
    flipud       = 0.0,
    degrees      = CONFIG['degrees'],
    scale        = CONFIG['scale'],
    hsv_h        = CONFIG['hsv_h'],
    hsv_s        = CONFIG['hsv_s'],
    hsv_v        = CONFIG['hsv_v'],
    close_mosaic = 10,
    project      = CONFIG['project'],
    name         = CONFIG['exp_name'],
    save         = True,
    save_period  = 10,
    exist_ok     = True,
    val          = True,
    plots        = True,
    amp          = True,
    verbose      = True,
)

BEST_WEIGHTS = f"{CONFIG['project']}/{CONFIG['exp_name']}/weights/best.pt"
LAST_WEIGHTS = f"{CONFIG['project']}/{CONFIG['exp_name']}/weights/last.pt"
print(f'\nTraining hoan tat!')
print(f'Best: {BEST_WEIGHTS}')

## Phan 10 -- Xem bieu do training

In [ ]:
save_dir = Path(CONFIG['project']) / CONFIG['exp_name']
plots = [
    ('results.png',          'Loss & Metrics theo epoch'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png',         'Precision-Recall Curve'),
    ('F1_curve.png',         'F1-Confidence Curve'),
]
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, (fname, title) in zip(axes.flatten(), plots):
    fpath = save_dir / fname
    if fpath.exists():
        ax.imshow(plt.imread(str(fpath)))
        ax.set_title(title, fontsize=11)
    else:
        ax.text(0.5, 0.5, f'Chua co: {fname}',
                ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Phan 11 -- Danh gia tren Test set

In [ ]:
model_eval = YOLO(BEST_WEIGHTS)
print('Danh gia tren TEST SET...')
print('='*55)

metrics = model_eval.val(
    data    = DATA_YAML,
    split   = 'test',
    imgsz   = CONFIG['img_size'],
    conf    = 0.25,
    iou     = 0.6,
    plots   = True,
    verbose = False,
)

map50 = metrics.box.map50
print(f'mAP@0.5      : {map50:.4f}  {"OK >= 0.85" if map50>=0.85 else "CHUA DAT < 0.85"}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')
print()
print('Per-class mAP@0.5:')
for i,(name,ap) in enumerate(zip(CONFIG['class_names'], metrics.box.ap50)):
    bar = '#' * int(ap*20)
    ok = 'OK' if ap>=0.80 else '--'
    print(f'  [{ok}] [{i}] {name:8s}: {ap:.4f}  |{bar:<20}|')

## Phan 12 -- Inference tren anh don le

In [ ]:
from google.colab import files

print('Upload anh test (jpg/png):')
uploaded = files.upload()

if uploaded:
    test_path = list(uploaded.keys())[0]
    model_infer = YOLO(BEST_WEIGHTS)
    results = model_infer.predict(
        source=test_path, conf=0.25, iou=0.45,
        imgsz=CONFIG['img_size'], save=True, verbose=False
    )
    result = results[0]
    plt.figure(figsize=(12,8))
    plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title('Ket qua Detection'); plt.show()

    boxes = result.boxes
    counts = {}
    for box in boxes:
        name = CONFIG['class_names'][int(box.cls.item())]
        conf = float(box.conf.item())
        counts[name] = counts.get(name,0)+1
        print(f'  [{name}] conf={conf:.2f}')
    print(f'Tong: {len(boxes)} phuong tien | {counts}')

## Phan 13 -- Inference tren Video

In [ ]:
from google.colab import files

print('Upload video test (mp4/avi):')
uploaded_v = files.upload()

if uploaded_v:
    vid_path = list(uploaded_v.keys())[0]
    cap = cv2.VideoCapture(vid_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    n_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f'Video: {vid_path} | FPS={fps:.1f} | Frames={n_f}')

    model_vid = YOLO(BEST_WEIGHTS)
    outs = model_vid.predict(
        source=vid_path, conf=0.25, iou=0.45,
        imgsz=CONFIG['img_size'], save=True,
        project='/content/vid_out', name='result',
        exist_ok=True, verbose=False, stream=True
    )
    frame_count=0; t0=time.time()
    for r in tqdm(outs, total=n_f, desc='Processing'):
        frame_count+=1
    avg_fps = frame_count/(time.time()-t0)
    print(f'Hoan tat: {frame_count} frames | {avg_fps:.1f} FPS')

    out_files = (list(Path('/content/vid_out/result').glob('*.mp4')) +
                 list(Path('/content/vid_out/result').glob('*.avi')))
    if out_files:
        print(f'Da luu: {out_files[0]}')
        files.download(str(out_files[0]))

## Phan 14 -- So sanh Baseline (YOLOv8n / s / m)

> Chay cell nay de co bang so sanh cho bao cao de tai.  
> Moi model train 30 epochs de so sanh nhanh.

In [ ]:
import pandas as pd

COMPARE_MODELS = ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']
CMP_EPOCHS     = 30
compare_results = []

for model_name in COMPARE_MODELS:
    print(f'\nTraining: {model_name}')
    m = YOLO(model_name)
    exp = f'cmp_{Path(model_name).stem}'
    m.train(
        data=DATA_YAML, epochs=CMP_EPOCHS,
        batch=CONFIG['batch_size'], imgsz=CONFIG['img_size'],
        device=device, project='/content/runs/compare', name=exp,
        exist_ok=True, verbose=False, plots=False, amp=True, workers=2
    )
    vm = m.val(data=DATA_YAML, split='test', verbose=False)
    dummy = np.random.randint(0,255,(640,640,3),dtype=np.uint8)
    _ = m.predict(dummy, verbose=False)
    t0=time.time()
    for _ in range(30): m.predict(dummy, verbose=False)
    fps_val = 30/(time.time()-t0)
    params = sum(p.numel() for p in m.model.parameters())/1e6
    compare_results.append({
        'Model': model_name,
        'mAP@0.5': round(vm.box.map50,4),
        'mAP@0.5:0.95': round(vm.box.map,4),
        'Precision': round(vm.box.mp,4),
        'Recall': round(vm.box.mr,4),
        'FPS': round(fps_val,1),
        'Params(M)': round(params,2),
    })
    print(f'  mAP@0.5={vm.box.map50:.4f} | FPS={fps_val:.1f} | Params={params:.1f}M')

df = pd.DataFrame(compare_results).sort_values('mAP@0.5',ascending=False)
print('\nBANG SO SANH BASELINE:')
print(df.to_string(index=False))
df.to_csv('/content/baseline_comparison.csv', index=False)

# Bieu do
fig, axes = plt.subplots(1,3,figsize=(15,5))
models_lbl = [Path(r['Model']).stem for r in compare_results]
colors = ['#378ADD','#1D9E75','#BA7517']

axes[0].bar(models_lbl, df['mAP@0.5'].tolist(), color=colors)
axes[0].axhline(0.85, color='red', ls='--', label='Muc tieu 0.85')
axes[0].set_title('mAP@0.5'); axes[0].legend(); axes[0].set_ylim(0,1)
for i,v in enumerate(df['mAP@0.5']): axes[0].text(i,v+0.01,f'{v:.3f}',ha='center')

axes[1].bar(models_lbl, df['FPS'].tolist(), color=colors)
axes[1].axhline(15, color='red', ls='--', label='Muc tieu 15 FPS')
axes[1].set_title('FPS'); axes[1].legend()

axes[2].bar(models_lbl, df['Params(M)'].tolist(), color=colors)
axes[2].set_title('So tham so (M)')

plt.suptitle(f'So sanh Baseline ({CMP_EPOCHS} epochs)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/baseline_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Da luu: baseline_comparison.csv va baseline_comparison.png')

## Phan 15 -- Luu ket qua vao Google Drive

In [ ]:
DRIVE_SAVE = Path(CONFIG['drive_data_path']) / 'training_results'
DRIVE_SAVE.mkdir(parents=True, exist_ok=True)

# Luu weights
drive_w = DRIVE_SAVE / 'weights'; drive_w.mkdir(exist_ok=True)
for wf in Path(CONFIG['project']).rglob('*.pt'):
    shutil.copy2(wf, drive_w / wf.name)
    print(f'Saved: {drive_w/wf.name}')

# Luu plots
drive_p = DRIVE_SAVE / 'plots'; drive_p.mkdir(exist_ok=True)
for img in (Path(CONFIG['project'])/CONFIG['exp_name']).glob('*.png'):
    shutil.copy2(img, drive_p/img.name)

# Luu yaml
shutil.copy2(DATA_YAML, DRIVE_SAVE/'dataset.yaml')

print(f'\nDa luu tat ca vao Drive: {DRIVE_SAVE}')
print('  weights/')
print('    best.pt  <- dung de inference')
print('    last.pt')
print('  plots/')
print('  dataset.yaml')

## Phan 16 -- Resume Training (tiep tuc neu bi ngat)

> Dung cell nay neu Colab bi ngat giua chung.

In [ ]:
RESUME_PATH = str(
    Path(CONFIG['drive_data_path']) / 'training_results' / 'weights' / 'last.pt'
)

if Path(RESUME_PATH).exists():
    print(f'Tim thay checkpoint: {RESUME_PATH}')
    print('Dang resume training...')
    model_r = YOLO(RESUME_PATH)
    results_r = model_r.train(resume=True)
    print('Resume hoan tat!')
else:
    print(f'Khong tim thay: {RESUME_PATH}')
    print('Hay chay training tu dau (Phan 9).')